# 02 - Integração e limpeza dos dados

União dos dois períodos de focos de queimadas (GeoJSON e CSV), limpeza de inconsistências e valores ausentes, junção com o índice RONI e geração das tabelas mensais integradas com a área queimada.

## 1. Leitura das bases coletadas

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

df_geojson = gpd.read_file("../dados/bdqueimadas_2024-11-30_2025-11-30.geojson").drop(columns=["geometry"])
df_csv = pd.read_csv("../dados/bdqueimadas_2025-12-01_2026-11-19.csv", encoding="utf-8-sig")
df_roni = pd.read_csv("../dados/roni.csv")
df_mb_brasil = pd.read_csv("../dados/mapbiomas_area_queimada_brasil.csv")
df_mb_biomas = pd.read_csv("../dados/mapbiomas_area_queimada_biomas.csv")
df_mb_estados = pd.read_csv("../dados/mapbiomas_area_queimada_estados.csv")

print("GeoJSON:", df_geojson.shape, "| CSV:", df_csv.shape, "| RONI:", df_roni.shape)

GeoJSON: (858405, 12) | CSV: (780217, 12) | RONI: (921, 3)


## 2. União dos dois períodos de focos

In [2]:
df_geojson["origem_formato"] = "GeoJSON (2024-2025)"
df_csv["origem_formato"] = "CSV (2025-2026)"

df = pd.concat([df_geojson, df_csv], ignore_index=True)
print("Total concatenado:", df.shape)
print("\nValores ausentes antes da limpeza:")
print(df.isna().sum())

Total concatenado: (1638622, 13)

Valores ausentes antes da limpeza:
DataHora               0
Satelite               0
Pais                   0
Estado                 0
Municipio              0
Bioma                  1
DiaSemChuva       149263
Precipitacao      149263
RiscoFogo         149263
FRP                    0
Latitude               0
Longitude              0
origem_formato         0
dtype: int64


## 3. Tipos de atributos

| Atributo | Tipo | Subtipo |
| :--- | :--- | :--- |
| `Satelite`, `Pais`, `Estado`, `Municipio`, `Bioma`, `origem_formato` | Qualitativo | Nominal |
| `ano_mes` | Qualitativo | Ordinal |
| `DiaSemChuva`, `ano`, `mes` | Quantitativo | Discreto |
| `DataHora`, `Precipitacao`, `RiscoFogo`, `FRP`, `Latitude`, `Longitude`, `roni`, `area_queimada_ha` | Quantitativo | Contínuo |

## 4. Padronização de datas e recorte de 12 x 12 meses

Limpamos o caractere `\ufeff` presente em 8 linhas intermediárias do CSV, convertemos `DataHora` para `datetime` e retiramos o dia avulso `2024-11-30` (`5.002` focos) para manter 12 meses em cada ciclo (`2024-12` a `2025-11` e `2025-12` a `2026-11`).

In [3]:
df["DataHora"] = df["DataHora"].astype(str).str.replace("\ufeff", "", regex=False)
df["DataHora"] = pd.to_datetime(df["DataHora"], format="mixed")

print("Intervalo de datas por arquivo:")
display(df.groupby("origem_formato")["DataHora"].agg(["min", "max"]))

df["ano"] = df["DataHora"].dt.year
df["mes"] = df["DataHora"].dt.month
df["ano_mes"] = df["DataHora"].dt.strftime("%Y-%m")

antes = len(df)
df = df[df["ano_mes"] >= "2024-12"].copy()
print("Focos removidos em 2024-11-30:", antes - len(df))
print("Total após recorte:", len(df))

Intervalo de datas por arquivo:


,min,max
origem_formato,,
CSV (2025-2026),2025-12-01 04:23:00,2026-11-18 18:53:00
GeoJSON (2024-2025),2024-11-30 03:46:00,2025-11-30 19:04:00


Focos removidos em 2024-11-30: 5002
Total após recorte: 1633620


## 5. Padronização de texto, duplicatas, valores `-999` e imputação

Outubro e novembro de 2026 concentram quase 50 mil focos sem medição meteorológica. Preenchemos `DiaSemChuva`, `Precipitacao` e `RiscoFogo` pela mediana com `SimpleImputer` para preservar esses focos na estação seca.

In [4]:
df["Estado"] = df["Estado"].str.strip().str.upper()
df["Bioma"] = df["Bioma"].str.strip()

print("Duplicatas exatas:", df.duplicated().sum())
print("Contradições (DataHora, Latitude, Longitude):", df.duplicated(subset=["DataHora", "Latitude", "Longitude"]).sum())

df = df.drop_duplicates()
df = df.drop_duplicates(subset=["DataHora", "Latitude", "Longitude"])

df[["DiaSemChuva", "RiscoFogo"]] = df[["DiaSemChuva", "RiscoFogo"]].replace(-999, np.nan)
df = df.dropna(subset=["Bioma"]).copy()

colunas_clima = ["DiaSemChuva", "Precipitacao", "RiscoFogo"]
imputer = SimpleImputer(strategy="median")
df[colunas_clima] = imputer.fit_transform(df[colunas_clima])

print("Nulos restantes:", df.isna().sum().sum())
print("Dimensões limpas:", df.shape)

Duplicatas exatas: 0
Contradições (DataHora, Latitude, Longitude): 4


Nulos restantes: 0
Dimensões limpas: (1633615, 16)


## 6. Junção com o índice RONI e salvamento da base de focos

A série de `roni.csv` termina em `ASO 2026` (mês central `2026-09`), deixando outubro e novembro de 2026 sem valor de `roni`.

In [5]:
mes_central = {
    "DJF": "01", "JFM": "02", "FMA": "03", "MAM": "04",
    "AMJ": "05", "MJJ": "06", "JJA": "07", "JAS": "08",
    "ASO": "09", "SON": "10", "OND": "11", "NDJ": "12",
}
df_roni["ano_mes"] = df_roni["ano"].astype(str) + "-" + df_roni["trimestre"].map(mes_central)
roni_mensal = df_roni[["ano_mes", "trimestre", "roni"]]

df = pd.merge(df, roni_mensal, on="ano_mes", how="left")
df.to_parquet("../dados/queimadas_limpo.parquet", index=False)
print("Salvo ../dados/queimadas_limpo.parquet:", df.shape)

Salvo ../dados/queimadas_limpo.parquet: (1633615, 18)


## 7. Geração das tabelas mensais integradas (`integrado_mensal_*.csv`)

Agregamos os focos limpos por mês e unimos o resultado com `roni.csv` e `mapbiomas_area_queimada_{brasil,biomas,estados}.csv`.

In [6]:
def agregar_focos(dados, chaves):
    return dados.groupby(chaves, as_index=False).agg(
        total_focos=("FRP", "count"),
        media_dias_sem_chuva=("DiaSemChuva", "mean"),
        media_precipitacao=("Precipitacao", "mean"),
        media_risco_fogo=("RiscoFogo", "mean"),
        media_frp=("FRP", "mean"),
        soma_frp=("FRP", "sum"),
    )

cols_focos = ["total_focos", "media_dias_sem_chuva", "media_precipitacao", "media_risco_fogo", "media_frp", "soma_frp"]

mensal_brasil = agregar_focos(df, ["ano_mes"])
mensal_brasil = pd.merge(mensal_brasil, roni_mensal, on="ano_mes", how="left")
mensal_brasil = pd.merge(mensal_brasil, df_mb_brasil[["ano_mes", "area_queimada_ha"]], on="ano_mes", how="left")

df_mb_biomas["Bioma"] = df_mb_biomas["territorio"].str.strip()
mensal_biomas = pd.merge(
    agregar_focos(df, ["ano_mes", "Bioma"]),
    df_mb_biomas[["ano_mes", "Bioma", "area_queimada_ha"]],
    on=["ano_mes", "Bioma"],
    how="outer",
)
mensal_biomas[cols_focos] = mensal_biomas[cols_focos].fillna(0)
mensal_biomas = pd.merge(mensal_biomas, roni_mensal, on="ano_mes", how="left")
mensal_biomas = mensal_biomas.sort_values(["ano_mes", "Bioma"]).reset_index(drop=True)

df_mb_estados["Estado"] = df_mb_estados["territorio"].str.strip().str.upper()
mensal_estados = pd.merge(
    agregar_focos(df, ["ano_mes", "Estado"]),
    df_mb_estados[["ano_mes", "Estado", "area_queimada_ha"]],
    on=["ano_mes", "Estado"],
    how="outer",
)
mensal_estados[cols_focos] = mensal_estados[cols_focos].fillna(0)
mensal_estados = pd.merge(mensal_estados, roni_mensal, on="ano_mes", how="left")
mensal_estados = mensal_estados.sort_values(["ano_mes", "Estado"]).reset_index(drop=True)

mensal_brasil.to_csv("../dados/integrado_mensal_brasil.csv", index=False)
mensal_biomas.to_csv("../dados/integrado_mensal_biomas.csv", index=False)
mensal_estados.to_csv("../dados/integrado_mensal_estados.csv", index=False)

print("Nulos após integração mensal (Brasil):")
print(mensal_brasil.isna().sum())
display(mensal_brasil.head())

Nulos após integração mensal (Brasil):
ano_mes                 0
total_focos             0
media_dias_sem_chuva    0
media_precipitacao      0
media_risco_fogo        0
media_frp               0
soma_frp                0
trimestre               2
roni                    2
area_queimada_ha        1
dtype: int64


,ano_mes,total_focos,media_dias_sem_chuva,media_precipitacao,media_risco_fogo,media_frp,soma_frp,trimestre,roni,area_queimada_ha
0,2024-12,59278,8.758679,0.150247,0.956117,9.405555,557542.5,NDJ,-1.05,1097364.98
1,2025-01,17354,5.633283,1.957289,0.512153,8.632650,149811.0,DJF,-1.10,684858.71
2,2025-02,11921,4.234796,1.549300,0.413548,8.225854,98060.4,JFM,-0.85,119853.85
3,2025-03,13727,6.992424,1.214493,0.539076,8.256626,113338.7,FMA,-0.66,111370.20
4,2025-04,7933,4.591958,1.304596,0.530904,6.894996,54698.0,MAM,-0.50,69155.78
